# ROS08 · ros2_control - hardware interfaces, controllers and the control loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS08_ros2_control_hardware_interfaces_controllers_and.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 70 min

**Skill Hub concepts:** ros2_control hardware interface

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS08")

## Map: the control loop the controller manager runs at `update_rate`

```
 every period:   hardware.read()  ──>  controllers.update()  ──>  hardware.write()
                 state interfaces       read states,               command interfaces
                 (position, velocity)   compute commands           (position, velocity or effort)

 controller_manager (ros2_control_node)
   ├── resource manager: loads the hardware plugin named in the URDF <ros2_control> tag
   │      mock_components/GenericSystem (sim) or your SystemInterface plugin (real drives over CAN, EtherCAT...)
   └── controllers (plugins, configured from YAML, switched at runtime with `spawner`):
          joint_state_broadcaster          states -> /joint_states
          forward_position_controller      /forward_position_controller/commands -> position commands
          joint_trajectory_controller      trajectories with interpolation (used by MoveIt)
```

| Your C++ `hardware_interface::SystemInterface` implements | When |
|---|---|
| `on_init` | parse URDF parameters (CAN IDs, gear ratios) |
| `export_state_interfaces` / `export_command_interfaces` | announce what can be read and commanded |
| `on_configure` / `on_activate` | open the bus, enable drives (lifecycle, lab ROS06) |
| `read(time, period)` / `write(time, period)` | every cycle, in the real-time thread: no blocking, no allocation |

Swapping simulation for hardware means changing one `<plugin>` line; controllers and the rest stay the same.

In [ ]:
from roboprep import ros
ros.ensure_ros(extras=("control",))

## Task 1: the loop itself, in plain Python
`hw` has `read()`, `write()`, `state` and `command` dicts. Write a controller that moves each joint's position command
toward its target at no more than `max_vel` rad/s, and the manager loop that calls read, update, write in that order.

In [ ]:
class RateLimitedPosition:
    def __init__(self, joints, max_vel):
        self.joints, self.max_vel, self.target = joints, max_vel, {j: 0.0 for j in joints}

    def update(self, state, command, dt):
        raise NotImplementedError  # TODO

def run_loop(hw, controller, rate_hz, seconds):
    raise NotImplementedError  # TODO

lab.check("loop", RateLimitedPosition, run_loop)

In [ ]:
%%writefile ros08_loop.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/ros08.h
// Grade it with lab.check("loop", "ros08_loop.c"). For C++, name the file .cpp in both places.
#include "ros08.h"

void rlp_init(RateLimitedPosition *c, size_t n_joints, double max_vel) {
    c->n = n_joints < RLP_MAX_JOINTS ? n_joints : RLP_MAX_JOINTS;
    c->max_vel = max_vel;
    for (size_t j = 0; j < c->n; ++j) c->target[j] = 0.0;
}

void rlp_update(RateLimitedPosition *c, const double *state, double *command, double dt) {
    (void)c; (void)state; (void)command; (void)dt; /* TODO */
}

void run_loop(const LoopOps *ops, double rate_hz, double seconds) {
    (void)ops; (void)rate_hz; (void)seconds; /* TODO: read, update, write each period */
}

In [ ]:
# lab.check("loop", "ros08_loop.c")  # uncomment after writing the C/C++ cell above

## Task 2: describe the hardware in the URDF
Add a `<ros2_control name="arm" type="system">` block using `mock_components/GenericSystem`, with joints `shoulder`
and `elbow`, each with a `position` command interface and `position` (initial value 0.0) and `velocity` state
interfaces.

In [ ]:
%%writefile arm_control.urdf
<?xml version="1.0"?>
<robot name="arm2">
  <link name="base_link"/><link name="link1"/><link name="link2"/>
  <joint name="shoulder" type="revolute"><parent link="base_link"/><child link="link1"/><origin xyz="0 0 0.1"/>
    <axis xyz="0 0 1"/><limit lower="-3.14" upper="3.14" effort="10" velocity="2"/></joint>
  <joint name="elbow" type="revolute"><parent link="link1"/><child link="link2"/><origin xyz="0.3 0 0"/>
    <axis xyz="0 0 1"/><limit lower="-3.14" upper="3.14" effort="10" velocity="2"/></joint>
  <!-- TODO: the <ros2_control> block -->
</robot>

## Task 3: configure the controllers

In [ ]:
%%writefile controllers.yaml
controller_manager:
  ros__parameters:
    update_rate: 100
    # TODO: declare joint_state_broadcaster and forward_position_controller with their types,
    #       then give forward_position_controller its joints and interface_name

Bring it up the way a launch file would: robot_state_publisher (publishes `/robot_description`), the controller
manager, then the spawner that loads and activates both controllers.

In [ ]:
import time
def bring_up():
    ros.start('ros2 run robot_state_publisher robot_state_publisher --ros-args -p robot_description:="$(cat arm_control.urdf)"', log="rsp.log")
    ros.start("ros2 run controller_manager ros2_control_node --ros-args --params-file controllers.yaml "
              "-r /controller_manager/robot_description:=/robot_description", log="cm.log")
    time.sleep(3)
    ros.ros("timeout 30 ros2 run controller_manager spawner joint_state_broadcaster forward_position_controller 2>&1 | tail -2", timeout=40)

bring_up()
ros.ros("ros2 control list_controllers; ros2 control list_hardware_interfaces")
ros.ros('ros2 topic pub --once /forward_position_controller/commands std_msgs/msg/Float64MultiArray "{data: [0.5, -0.3]}" > /dev/null; sleep 1; ros2 topic echo --once /joint_states | grep -A2 position', timeout=20)
ros.stop_all()

In [ ]:
lab.check("controllers")

## Checkpoint: Math
**M1 (Apply).** `update_rate: 500`. Control period in milliseconds?

**M2 (Apply).** Six joints, each with position and velocity state interfaces and a position command interface. Total
interfaces the hardware exports?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** Which belongs inside a controller's `update()`?
- a) a blocking service call to look up a parameter
- b) arithmetic on preallocated buffers using the latest states, writing the command interfaces
- c) logging every cycle to a file

**K2 (Understand).** Explain why ros2_control separates hardware interfaces from controllers, and what changes when
the Storm stepper boards replace `mock_components/GenericSystem`.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros08.h`).
3. In `roboprep/checks/ros08.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()